In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import tensor

import numpy as np
import pandas as pd
import networkx as nx

In [3]:
DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'

# Aims outline

Use same as first attempt but replace linear function with MML functon from LEMBAS

# Using network to filter input TFs to those relevant to the target node of a given model

In [46]:
#creating network for network TSV, obtain all nodes in all simple paths to the target not specified
def genes_in_path(graph, source_nodes, target_node):
    '''
    Paramaters
    -------------
    parameter : graph
        networkx graph
    parameter : soruce_nodes
        list of node labels you wish to use as source nodes
    target_node : string
        label of desired target node
    
    Returns
    -------------
    nodes : List
        List of node labels in the network that are present in at least on shortest path to the specified target node from any of the source nodes.
    '''
    #keep track of all unique nodes found in a shortest path
    node_set = set()
    for source_node in source_nodes:
        try:
            sps = nx.all_shortest_paths(graph, 
                            source=source_node, 
                            target=target_node,
                            weight = 'None')
        except:
            print('No path from {source_node} to {target_node}')

        for path_nodes in list(sps):
            node_set.update(set(path_nodes))

    return(node_set)


In [25]:
net = pd.read_csv(f"{DATA_ROOT}/Full data files/network(full).tsv", sep='\t')

In [26]:
network = nx.from_pandas_edgelist(net, 
                             source='TF', 
                             target='Gene', 
                             edge_attr='Interaction')

In [27]:
gene_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

In [28]:
TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)

In [29]:
not_in_network = []

for TF in list(TF_expressions.columns):
    if (TF in (network.nodes())) == False:
        not_in_network.append(TF)

for gene in list(gene_expressions.columns):
    if (gene in (network.nodes())) == False:
        not_in_network.append(gene)

In [ ]:
#subset to just the TFs that are in the network
TF_expressions = TF_expressions[[column for column in TF_expressions.columns if column in list(network.nodes())]]

In [47]:
genes_to_keep = genes_in_path(network, list(TF_expressions.columns), 'AAGAB')

NetworkXNoPath: Target AAGAB cannot be reached from given sources

### Creating dataset object

In [ ]:
#torch tutorial code to use accelerator when available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [ ]:
#DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
#TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)
#TF_expressions.shape

In [ ]:
from torch.utils.data import Dataset


class CustomTFGE(Dataset):
    def __init__(self, device, transform=None, target_transform=None):
        #load the two tsv files
        self.DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'
        self.TF_expressions = pd.read_csv((f"{self.DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)
        self.gene_expressions = pd.read_csv((f"{self.DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

        #no transforms needed so set to none
        self.transform = transform
        self.target_transform = target_transform

        #convert to torch tensors
        self.TF_expressions = torch.tensor(np.asarray(self.TF_expressions).T, dtype = torch.float32, device = device)
        self.gene_expressions = torch.tensor(np.asarray(self.gene_expressions), dtype = torch.float32, device = device)

    def __len__(self):
        #length of the dataset is the number of samples (not TFs in the dataset) - 15935
        return self.TF_expressions.shape[0]

    def __getitem__(self, idx):
        #in this case want to always retrieve the same gene (target) but a different sample containg all the TF values
        TFs_exp = self.TF_expressions[:, idx]
        Gene_exp = self.gene_expressions[:, 1]
        #returns TFs exp and Gene_exp 
        return TFs_exp, Gene_exp

In [ ]:
#initialise an instance of the dataset object - pass device so tensors and model on same device
dataset = CustomTFGE(device)
dataset

In [ ]:
#new way to create train and test dataset with pytorches dataset objects
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [ ]:
#MML like function from LEMBAS: https://github.com/Lauffenburger-Lab/LEMBAS/blob/main/Model/activationFunctions.py

######## MML activation
def MMLactivation(x, leak=0.01):
    #x[x<0] = leak*x[x<0]
    #x[x>=0.5] = 0.5 * (1 + (1./(0.5/(x[x>=0.5]-0.5) + 1)))
    x = np.where(x < 0, x * leak, x)
    x = np.where(x > 0.5, 1 - 0.25/x, x) #Pyhton will display division by zero warning since it evaluates both before selecting
    return x

In [ ]:
#defining the neural network
class BasicNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #self.flatten = nn.Flatten()
        self.leak = 0.01
        self.linear_layer = nn.Sequential(
            #linear activation layer takes 1198 TFs per sample -> outputs a value for the TF for all 15935 samples
            nn.Linear(1198, 15935)
        )

    def MMLactivation(self, x):
        xFilter = x<=0
        x[xFilter] = self.leak*x[xFilter]
        xFilter = x>0.5
        x[xFilter] = 1 - (0.25/x[xFilter])
        return x
    
    def forward(self, x):
        #forward pass is simply the linear layer
        expressions = self.MMLactivation(x)
        print(expressions.shape)
        expressions = self.linear_layer(x)
        return(expressions)

    

In [ ]:
#put model on same device as the tensors
model = BasicNeuralNetwork().to(device)
print(model)

BasicNeuralNetwork(
  (linear_layer): Sequential(
    (0): Linear(in_features=1198, out_features=15935, bias=True)
  )
)


## Train test loop

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    losses = []
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        
        loss = loss.item()
        print(f"loss: {loss:>7f}")
        losses.append(loss)
    return(losses)


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            print(f"MSE on test dataset is {loss_fn(pred, y).item()}")
            


In [ ]:
#intialise hyperparameters - batch size is number of samples so that each backprop is done with the entire dataset (gradient descent not stochastic gradient descent)
#dataset is small enough for this to be fine
#investigate hyperparam tuning later
learning_rate = 1e-3
batch_size = 15935
epochs = 500

#initialize MSE loss function - same as LEMBAS
loss_fn = nn.MSELoss()

#initialise same optimiser as LEMBAS
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    losses = train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
torch.Size([959, 1198])
loss: 6.277700
torch.Size([239, 1198])
MSE on test dataset is 6.277544021606445
Epoch 2
-------------------------------
torch.Size([959, 1198])
loss: 6.277701
torch.Size([239, 1198])
MSE on test dataset is 6.277544021606445
Epoch 3
-------------------------------
torch.Size([959, 1198])
loss: 6.277700
torch.Size([239, 1198])
MSE on test dataset is 6.2775444984436035
Epoch 4
-------------------------------
torch.Size([959, 1198])
loss: 6.277700
torch.Size([239, 1198])
MSE on test dataset is 6.277544021606445
Epoch 5
-------------------------------
torch.Size([959, 1198])
loss: 6.277700
torch.Size([239, 1198])
MSE on test dataset is 6.277544021606445
Epoch 6
-------------------------------
torch.Size([959, 1198])
loss: 6.277701
torch.Size([239, 1198])
MSE on test dataset is 6.277544021606445
Epoch 7
-------------------------------
torch.Size([959, 1198])
loss: 6.277700
torch.Size([239, 1198])
MSE on test dataset is 6.2775444

# Saving model

In [ ]:
torch.save(model, 'models/first_attempt_single_target_model.pth')

In [ ]:
#How to load for reference
model = torch.load('models/first_attempt_single_target_model.pth', weights_only=False)